<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-misc/audio_feature_extracor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [ ]:
# model_id = "openai/whisper-large-v3-turbo"
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=16,  # batch size for inference - set based on your device
    torch_dtype=torch_dtype,
    device=device,
)

In [ ]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sample = dataset[0]["audio"]
print(sample)

In [ ]:
result = pipe(sample)
print(result["text"])

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_voice(filename='recording.wav'):
  js = Javascript('''
    async function recordAudio() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Record';
      button.style.background = 'red';
      button.style.color = 'white';
      button.style.padding = '10px';
      document.body.appendChild(div);
      div.appendChild(button);

      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks);
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.callback(reader.result);
        };
      };

      button.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          button.textContent = 'Stop Recording';
        } else {
          recorder.stop();
          button.textContent = 'Done!';
        }
      };

      return new Promise((resolve) => {
        window.callback = resolve;
      });
    }
  ''')
  display(js)
  data = output.eval_js('recordAudio()')
  binary = base64.b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# Run the function
audio_file = record_voice()
print(f"Saved as {audio_file}")

In [ ]:
from IPython.display import Audio
Audio(audio_file)

In [ ]:
generate_kwargs = {
    "language": "english",
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

# result = pipe(sample, return_timestamps=True)
result = pipe('recording.wav', generate_kwargs=args)
print(result["text"])

In [ ]:
#@title Part 1: Explicit Audio Chunking with WavLM Embeddings
import torch
import gc
import numpy as np
from transformers import AutoFeatureExtractor, WavLMModel
from datasets import load_dataset

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
# Note: WavLM is typically stable in float32; float16 can be used on modern GPUs
torch_dtype = torch.float32

model_id = "microsoft/wavlm-base-plus"

# 2. Load WavLM Native Components
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = WavLMModel.from_pretrained(model_id).to(device, dtype=torch_dtype)

# 3. Pull the sample long audio array
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sr = 16000
audio_array = dataset[0]["audio"]["array"]

# 4. Define Chunking and Stride rules (matching your original spec)
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sr
overlap_samples = OVERLAP_DURATION * sr
stride_samples = chunk_samples - overlap_samples

chunked_embeddings = []

print(f"Total audio length: {len(audio_array)/sr:.2f} seconds")
print("Extracting acoustic chunk embeddings manually...\n")

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    global_start_time = start_idx / sr
    global_end_time = min(end_idx / sr, len(audio_array) / sr)

    if len(chunk) < sr * 0.5: # Skip tiny leftovers
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    # Extract structural acoustic features
    inputs = feature_extractor(chunk, sampling_rate=sr, return_tensors="pt")
    input_values = inputs.input_values.to(device, dtype=torch_dtype)

    with torch.no_grad():
        outputs = model(input_values)
        # Sequence shape: [batch, sequence_length, 768]
        last_hidden_states = outputs.last_hidden_state

        # Mean Pooling: Collapse time dimension to catch holistic properties of the window
        mean_pooled = torch.mean(last_hidden_states, dim=1).squeeze()
        embedding = mean_pooled.cpu().numpy()

    chunked_embeddings.append({
        "chunk_index": len(chunked_embeddings),
        "global_window_seconds": (global_start_time, global_end_time),
        "embedding": embedding # 768-dimensional float array
    })

    print(f"Processed Chunk {chunked_embeddings[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s | Shape: {embedding.shape}")

# To represent the entire track as a single, holistic master vector:
all_vectors = [c["embedding"] for c in chunked_embeddings]
master_audio_embedding = np.mean(all_vectors, axis=0)
print(f"\nFinal global track embedding generated with shape: {master_audio_embedding.shape}")

In [ ]:
#@title Part 2: Measuring Similarity (Qualitative vs. Quantitative)
import torch.nn.functional as F

# Helper function to extract a pooled embedding vector from raw audio
def get_holistic_embedding(audio_data, sampling_rate=16000):
    inputs = feature_extractor(audio_data, sampling_rate=sampling_rate, return_tensors="pt")
    inputs = inputs.input_values.to(device, dtype=torch_dtype)
    with torch.no_grad():
        outputs = model(inputs)
        return torch.mean(outputs.last_hidden_state, dim=1) # Keeps batch dim for F.cosine_similarity

# --- Scenario A: Qualitative Similarity Example ---
# Two audio slices from the exact same speaker, recording environment, and background noise levels,
# but they are saying entirely different sentences.
audio_speaker1_phraseA = audio_array[0 : 10 * sr]             # First 10 seconds
audio_speaker1_phraseB = audio_array[15 * sr : 25 * sr]       # A completely different 10 seconds

emb_qual_1 = get_holistic_embedding(audio_speaker1_phraseA)
emb_qual_2 = get_holistic_embedding(audio_speaker1_phraseB)

qualitative_similarity = F.cosine_similarity(emb_qual_1, emb_qual_2).item()


# --- Scenario B: Quantitative Similarity Example ---
# Two audio clips with matching cadence properties. For demonstration, we simulate
# a quantitative variation (like an exact pitch shift or clean speed alteration)
# to show how structural audio manipulation preserves high quantitative feature overlaps.
audio_base = audio_array[0 : 15 * sr]

# Simulate a clean quantitative shift (e.g., applying a minor pitch modulation natively)
# For code safety without extra external dependencies, we use a basic array operation
audio_pitched = np.ascontiguousarray(audio_base * 0.95)

emb_quant_1 = get_holistic_embedding(audio_base)
emb_quant_2 = get_holistic_embedding(audio_pitched)

quantitative_similarity = F.cosine_similarity(emb_quant_1, emb_quant_2).item()


# --- Print Metric Diagnostics ---
print("\n--- EMBEDDING SIMILARITY ANALYSIS ---")
print(f"Qualitative Match (Same Speaker / Context, Different Words): {qualitative_similarity:.4f}")
print(f"Quantitative Match (Same Pacing / Signal Structure Alteration): {quantitative_similarity:.4f}")

In [ ]:
#@title DLAI Suggestion for Interview Rating pipeline
import torch
import numpy as np
from transformers import pipeline

# =====================================================================
# STEP 1: AUDIO TRANSCRIPTION & TIME ANALYTICS (Using Whisper)
# =====================================================================

# Initialize Whisper pipeline with chunking and timestamp options enabled
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device=device
)

def analyze_interview_audio(audio_path_or_array):
    print("Processing audio with Whisper...")
    # Retrieve transcription with word-level timestamps
    result = pipe(
        audio_path_or_array,
        return_timestamps="word"
    )

    text = result["text"]
    chunks = result["chunks"]

    # Programmatically calculate delivery metrics
    total_words = len([c for c in chunks if "text" in c])

    # Calculate silences/pauses (gaps between words greater than 1.5 seconds)
    pauses = 0
    for i in range(len(chunks) - 1):
        end_current = chunks[i]["timestamp"][1]
        start_next = chunks[i+1]["timestamp"][0]
        if end_current is not None and start_next is not None:
            if (start_next - end_current) > 1.5:
                pauses += 1

    total_duration = chunks[-1]["timestamp"][1] if chunks else 1.0
    words_per_minute = (total_words / total_duration) * 60

    return {
        "transcript": text,
        "metrics": {
            "words_per_minute": round(words_per_minute, 1),
            "long_pauses_count": pauses,
            "total_duration_seconds": round(total_duration, 2)
        }
    }

# Simulating processing on a dummy sample
# sample_audio = "candidate_answer.mp3"
# audio_analysis = analyze_interview_audio(sample_audio)

# Mocked output for the sake of the structural demonstration:
audio_analysis = {
    "transcript": "A REST API is stateless... um, meaning that the server does not store any session data about the client. Every request must, uh, contain all the information needed.",
    "metrics": {
        "words_per_minute": 110.5,
        "long_pauses_count": 2,
        "total_duration_seconds": 15.4
    }
}

print("\n--- Audio Analytics Extracted ---")
print(audio_analysis)

# =====================================================================
# STEP 2: KNOWLEDGE VERIFICATION (LLM-as-a-Judge Prompt)
# =====================================================================

# This is how you would construct your prompt to a foundational LLM
# to keep it grounded, objective, and outputting structured JSON.

reference_answer_from_rag = """
A REST API must be stateless. The server should not store any context or session data about the client.
Each individual request from a client must contain all necessary information and authentication details to understand and process it.
"""

llm_judge_prompt = f"""
You are an expert technical interviewer acting as a strict, objective grading judge.
Analyze the candidate's transcript against the Reference Answer retrieved from our Knowledge Base.

[Reference Answer]
{reference_answer_from_rag}

[Candidate Transcript]
{audio_analysis['transcript']}

[Grading Instructions]
1. Evaluate if the logical sequence matches and if the fundamental points are fully covered.
2. To prevent hallucination or being overly generous, you MUST extract a direct quote from the Candidate Transcript to prove a point was met.
3. Ignore minor verbal fillers like "um" or "uh" (focus entirely on semantic correctness).

Provide your final assessment strictly in the following JSON format:
{{
  "technical_accuracy_score": <int from 0 to 100>,
  "points_covered": [
     {{"point": "Statelessness criteria", "status": "Met/Unmet", "justifying_quote": "string or null"}}
  ],
  "logical_progression_rating": "Excellent/Fair/Poor",
  "constructive_feedback": "string"
}}
"""

print("\n--- Generated LLM-as-a-Judge System Prompt ---")
print(llm_judge_prompt)

In [ ]:
#@title Extracting Fluency and Utterances metrics from audio sample
import torch
from transformers import pipeline

# 1. Setup the pipeline with word-level timestamps explicitly enabled
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device=device
)

# 2. Process your interview audio segment
# result = asr_pipe("candidate_interview_clip.wav", return_timestamps="word")

# Mocking a raw word-level timestamp structure returned by Whisper for demonstration:
mock_whisper_output = {
    "text": "A REST API is um, stateless... meaning that, meaning that the server does not store data.",
    "chunks": [
        {"text": "A", "timestamp": (0.1, 0.3)},
        {"text": " REST", "timestamp": (0.3, 0.6)},
        {"text": " API", "timestamp": (0.6, 1.0)},
        {"text": " is", "timestamp": (1.0, 1.2)},
        {"text": " um,", "timestamp": (1.2, 1.6)},          # Filler utterance
        {"text": " stateless...", "timestamp": (3.8, 4.5)}, # Note the huge time jump here (2.2s pause)
        {"text": " meaning", "timestamp": (4.5, 4.8)},      # Repetition start
        {"text": " that,", "timestamp": (4.8, 5.0)},
        {"text": " meaning", "timestamp": (5.0, 5.3)},      # Repetition end
        {"text": " that", "timestamp": (5.3, 5.5)},
        {"text": " the", "timestamp": (5.5, 5.7)},
        {"text": " server", "timestamp": (5.7, 6.1)},
    ]
}

def calculate_qualitative_metrics(whisper_data):
    chunks = whisper_data["chunks"]

    total_words = len(chunks)
    duration = chunks[-1]["timestamp"][1] - chunks[0]["timestamp"][0]

    # 1. Calculate Pacing (WPM)
    wpm = (total_words / duration) * 60

    # 2. Track Hesitations (Pauses > 1.5 seconds)
    pauses_count = 0
    filler_utterances = 0
    words_list = []
    repetitions = 0

    # List of targeted filler tokens to monitor
    filler_dictionary = ["um", "uh", "ah", "like"]

    for i in range(len(chunks)):
        clean_word = chunks[i]["text"].strip().lower().replace(",", "").replace(".", "")
        words_list.append(clean_word)

        # Check for verbal tics/utterances
        if clean_word in filler_dictionary:
            filler_utterances += 1

        # Check for structural silences between words
        if i < len(chunks) - 1:
            current_word_end = chunks[i]["timestamp"][1]
            next_word_start = chunks[i+1]["timestamp"][0]

            if current_word_end and next_word_start:
                if (next_word_start - current_word_end) > 1.5:
                    pauses_count += 1

        # Basic check for quick structural phrase repeats (e.g., "meaning that, meaning that")
        if i >= 2:
            if words_list[i] == words_list[i-2] and words_list[i-1] == words_list[i-3]:
                repetitions += 1

    return {
        "pace_wpm": round(wpm, 1),
        "hesitation_pauses": pauses_count,
        "filler_utterance_count": filler_utterances,
        "phrase_repetitions": repetitions,
        "speech_fluidity_rating": "Fluid" if pauses_count == 0 and filler_utterances < 2 else "Fragmented"
    }

metrics = calculate_qualitative_metrics(mock_whisper_output)

print("--- QUALITATIVE AUDIO METRICS ---")
for key, val in metrics.items():
    print(f"{key.replace('_', ' ').title()}: {val}")

In [ ]:
!pip install librosa

In [ ]:
#@title Audio feature extractor
import librosa
import torch
import numpy as np
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# 1. Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load a pre-trained model fine-tuned to recognize tone, pacing, and speech emotion
# This model uses its internal 1D-CNN filters to automatically map acoustic signatures
# model_id = "ehsanaghaei/wav2vec2-base-Speech_Emotion_Recognition"
# model_id = "harshit345/xlsr-wav2vec2-speech-emotion-recognition" # fine-tuned model to extract sentiment from a audio clip
model_id = "microsoft/wavlm-large"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = AutoModelForAudioClassification.from_pretrained(model_id).to(device)

# 3. Simulate processing a candidate's response audio array (16kHz)
# (In production, replace this with your actual audio array segment)
sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)

# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# mock_audio_duration_seconds = 5
# dummy_audio = np.random.uniform(-0.1, 0.1, sampling_rate * mock_audio_duration_seconds).astype(np.float32)

# 4. Extract raw features using the model's preprocessing configuration
# inputs = feature_extractor(dummy_audio, sampling_rate=sampling_rate, return_tensors="pt")
inputs = feature_extractor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
input_values = inputs.input_values.to(device)

# 5. Pass through the pre-trained neural filters
with torch.no_grad():
    logits = model(input_values).logits

    # Apply Softmax to turn the raw outputs into percentage probabilities
    probabilities = torch.nn.functional.softmax(logits, dim=-1).squeeze().cpu().numpy()

# 6. Map probabilities to the learned vocal profiles
labels = model.config.id2label

print("--- AUTOMATIC ACOUSTIC ANALYSIS ---")
for idx, prob in enumerate(probabilities):
    label_name = labels.get(idx, f"Class {idx}")
    print(f"Confidence score for trait [{label_name}]: {prob * 100:.2f}%")

In [ ]:
#@title Audio feature extractor
import librosa
import torch
import numpy as np
from transformers import AutoFeatureExtractor, AutoModel

# 1. Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load a pre-trained model fine-tuned to recognize tone, pacing, and speech emotion
# This model uses its internal 1D-CNN filters to automatically map acoustic signatures
# model_id = "ehsanaghaei/wav2vec2-base-Speech_Emotion_Recognition"
model_id = "microsoft/wavlm-large"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)

# 3. Simulate processing a candidate's response audio array (16kHz)
# (In production, replace this with your actual audio array segment)
sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)
# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# 4. Extract raw features using the model's preprocessing configuration
# inputs = feature_extractor(dummy_audio, sampling_rate=sampling_rate, return_tensors="pt")
inputs = feature_extractor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
input_values = inputs.input_values.to(device)

# 5. Pass through the pre-trained neural filters
with torch.no_grad():
    outputs = model(input_values)

    # Extract the hidden states from the final layer of the transformer backbone
    # Shape: [batch_size, sequence_length, 1024]
    last_hidden_states = outputs.last_hidden_state

    # Perform mean pooling across the time (sequence) dimension to get a single vector
    embeddings = torch.mean(last_hidden_states, dim=1).squeeze().cpu().numpy()

print("--- VECTORIZATION COMPLETE ---")
print(f"Generated embedding vector with shape: {embeddings.shape}")
print(f'Embedding vector: {embeddings}')

In [9]:
#@title Acoustic feature extraction with chunking
import librosa
import torch
import numpy as np
from transformers import AutoFeatureExtractor, AutoModel

# 1. Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load a pre-trained model fine-tuned to recognize tone, pacing, and speech emotion
# This model uses its internal 1D-CNN filters to automatically map acoustic signatures
# model_id = "ehsanaghaei/wav2vec2-base-Speech_Emotion_Recognition"
model_id = "microsoft/wavlm-large"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)

# 3. Simulate processing a candidate's response audio array (16kHz)
# (In production, replace this with your actual audio array segment)
sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)
# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# Define chunking boundaries (e.g., 10 seconds per chunk)
chunk_duration = 10
chunk_size = sampling_rate * chunk_duration

all_embeddings = []

print("--- STARTING BATCH PROCESSING ---")
# Loop through the 30-minute audio in 10-second intervals
for i in range(0, len(audio_array), chunk_size):
    chunk = audio_array[i : i + chunk_size]

    # 4. Extract raw features using the model's preprocessing configuration
    # Preprocess the individual chunk
    inputs = feature_extractor(chunk, sampling_rate=sampling_rate, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    # 5. Pass through the pre-trained neural filters
    with torch.no_grad():
        outputs = model(input_values)
        last_hidden_states = outputs.last_hidden_state

        # Mean pool this specific chunk
        chunk_embedding = torch.mean(last_hidden_states, dim=1).squeeze().cpu().numpy()
        all_embeddings.append(chunk_embedding)

# Step 6: Average all chunks together to get one final 1024 vector for the entire 30 mins
final_embedding = np.mean(all_embeddings, axis=0)

print("--- COMPLETE ---")
print(f"Final aggregated embedding shape: {final_embedding.shape}")
print(f'Embedding vector: {final_embedding}')

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

--- STARTING BATCH PROCESSING ---
--- COMPLETE ---
Final aggregated embedding shape: (1024,)
Embedding vector: [-0.07426869  0.02633431  0.01944999 ...  0.01307706  0.06034928
  0.01776272]


In [ ]:
import torch
import gc
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

# 2. Load the native model components
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa"
).to(device)

In [2]:
sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)
# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# 4. Define your Chunking and Stride rules
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sampling_rate
overlap_samples = OVERLAP_DURATION * sampling_rate
stride_samples = chunk_samples - overlap_samples

# This structure will hold your explicit chunk-wise data
chunked_results = []

print(f"Total audio length: {len(audio_array)/sampling_rate:.2f} seconds")
print("Processing explicit chunks manually...\n")

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    # Calculate the global clock positioning for metadata records
    global_start_time = start_idx / sampling_rate
    global_end_time = min(end_idx / sampling_rate, len(audio_array) / sampling_rate)

    if len(chunk) < sampling_rate * 0.5: # Skip tiny leftover audio shards
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    print(f'Running the pipeline on chunk of length {len(chunk)}')
    result = asr_pipe(
        chunk,
        return_timestamps="word",
        generate_kwargs={"temperature": 0.0, "language": "en"},
    )

    # FIX 1: Generate BOTH input_features and attention_mask
    # inputs = processor(chunk, sampling_rate=sampling_rate, return_attention_mask=True, return_tensors="pt")
    # input_features = inputs.input_features.to(device, dtype=torch_dtype)
    # attention_mask = inputs.attention_mask.to(device)

    # Generate text & word timestamps for THIS SPECIFIC CHUNK ONLY
    # with torch.no_grad():
    #     # FIX 2: Set return_dict_in_generate=True so we can safely unpack outputs
    #     outputs = model.generate(
    #         input_features=input_features,
    #         attention_mask=attention_mask,
    #         return_timestamps='word',
    #         return_dict_in_generate=True, # Wraps outputs in a safe dictionary structure,
    #         temperature=0.0,
    #         language="en",
    #     )

    # # FIX 3: Unpack the generated text token ids cleanly
    # predicted_ids = outputs["sequences"]
    # transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Word-level timestamps come from processor's decode with offsets
    # decoded = processor.batch_decode(
    #     predicted_ids,
    #     skip_special_tokens=True,
    #     output_offsets=True   # <-- this gives you word-level start/end offsets
    # )[0]

    # word_offsets = decoded["offsets"]

    # words_global = [
    #     {
    #         "text": w["text"].strip(),
    #         "start": round(w["timestamp"][0] + global_start_time, 2),
    #         "end": round(w["timestamp"][1] + global_start_time, 2),
    #     }
    #     for w in word_offsets
    #     if w["timestamp"][0] is not None and w["timestamp"][1] is not None
    # ]

    # Pull word-level timestamps from segments instead of output_offsets
    # words_global = []
    # segments = outputs.get("segments")

    # if segments is not None:
    #     # segments[0] = list of segment dicts for batch item 0
    #     for seg in segments[0]:
    #         # 'result' holds the per-token/word timing info depending on transformers version
    #         seg_text = processor.decode(seg["tokens"], skip_special_tokens=True).strip()
    #         seg_start = seg["start"].item() if torch.is_tensor(seg["start"]) else seg["start"]
    #         seg_end = seg["end"].item() if torch.is_tensor(seg["end"]) else seg["end"]
    #         words_global.append({
    #             "text": seg_text,
    #             "start": round(seg_start + global_start_time, 2),
    #             "end": round(seg_end + global_start_time, 2),
    #         })

    transcription = result["text"].strip()
    words_global = [
        {
            "text": w["text"].strip(),
            "start": round(w["timestamp"][0] + global_start_time, 2),
            "end": round(w["timestamp"][1] + global_start_time, 2),
        }
        for w in result["chunks"]
        if w["timestamp"][0] is not None and w["timestamp"][1] is not None
    ]

    # Append the structured result for this explicit block
    chunked_results.append({
        "chunk_index": len(chunked_results),
        "global_window_seconds": (global_start_time, global_end_time),
        "text": transcription.strip(),
        "words": words_global,   # <-- new field
    })

    print(f"Processed Chunk {chunked_results[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s")

    # --- Memory hygiene: free GPU tensors each iteration ---
    # del inputs, input_features, attention_mask, outputs, predicted_ids
    del result, transcription, words_global
    gc.collect()
    torch.cuda.empty_cache()

# 6. Inspect your isolated chunk-wise data structure
print("\n--- VIEW OF MANUALLY SEPARATED CHUNKS ---")
import pprint
pprint.pprint(chunked_results)

NameError: name 'librosa' is not defined

In [1]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"Total: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")
!nvidia-smi

False
/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
all_word_timestamps = []

for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    global_start_time = start_idx / sampling_rate
    global_end_time = min(end_idx / sampling_rate, len(audio_array) / sampling_rate)

    if len(chunk) < sampling_rate * 0.5:
        continue

    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    inputs = processor(chunk, sampling_rate=sampling_rate, return_attention_mask=True, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch_dtype)
    attention_mask = inputs.attention_mask.to(device)

    with torch.no_grad():
        # outputs = model.generate(
        #     input_features=input_features,
        #     attention_mask=attention_mask,
        #     return_timestamps='word', # Keeps word-level logit markers active
        #     return_dict_in_generate=True,
        #     temperature=0.0
        # )
        # --- FIXED GENERATION KWARGS FOR WORD-LEVEL DETAILS ---
        outputs = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            return_dict_in_generate=True,

            # 1. Setting return_timestamps to True enables timestamp tracking
            return_timestamps=True,

            # 2. We use generation_config options to explicitly trigger word-level steps
            generation_config=model.generation_config,

            # 3. Giving it a tiny temperature variation allows the word-alignment
            # algorithm to find fine-grained timestamp paths instead of falling back to sentences
            temperature=0.0,

            # Some Whisper versions require setting the language to prevent default heuristics
            language="en",
            task="transcribe"
        )

    # --- CHANGED: Extracting timestamps cleanly ---
    predicted_ids = outputs["sequences"]
    decoded_structures = processor.batch_decode(predicted_ids, return_timestamps=True)
    chunk_outputs = decoded_structures[0]["chunks"]

    # 1. Use the tokenizer's internal timestamp decoder instead of batch_decode
    # This yields a structured list containing text strings and time tuples
    # chunk_outputs = processor.tokenizer._decode_with_timestamps(
    #     predicted_ids[0],
    #     return_timestamps="word"
    # )
    print(chunk_outputs)

    # 2. Iterate through tokens to calculate global time offsets
    for item in chunk_outputs:
        # Filter for dictionaries that contain valid word text and timestamps
        if isinstance(item, dict) and "text" in item and "timestamp" in item:
            word_text = item["text"].strip()

            if not word_text or item["timestamp"] is None:
                continue

            local_start, local_end = item["timestamp"]

            # CRITICAL OFFSET: Add the global window baseline to map onto the full file duration
            global_word_start = global_start_time + local_start
            global_word_end = global_start_time + local_end

            # --- OVERLAP HANDLING (DEDUPLICATION) ---
            # Because chunks overlap by 5s, ignore duplicate words falling in the overlap zone
            if len(all_word_timestamps) > 0:
                last_word_end = all_word_timestamps[-1]["end"]
                if global_word_start < last_word_end:
                    continue

            all_word_timestamps.append({
                "word": word_text,
                "start": round(global_word_start, 2),
                "end": round(global_word_end, 2)
            })

    print(f"Processed Chunk: Window {global_start_time:.1f}s to {global_end_time:.1f}s")

# 6. Inspect your absolute timeline of word-level timestamps
print("\n--- EXTRACTED WORD TIMESTAMPS (FIRST 20) ---")
import pprint
pprint.pprint(all_word_timestamps[:20])

In [23]:
decoded_structures[0]['chunks']

TypeError: string indices must be integers, not 'str'

In [ ]:
# 3. Instantiate the pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

# 4. Process the data sample's audio array (16kHz)
# sampling_rate = 16000
file_path = "audio_indian.mp3"
# audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)
# audio_array = audio_array.astype(np.float32)

# print(f"Total audio length: {len(audio_array)/sampling_rate:.2f} seconds")
print("Processing with internal pipeline chunking (Memory Safe)...\n")

# 5. Define generation kwargs for the model decoder
generate_kwargs = {
    "temperature": 0.0,
    "condition_on_prev_tokens": True,
    "compression_ratio_threshold": 2.4,
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
}

# 6. Execute the entire audio array through the pipeline natively
# chunk_length_s=30 and stride_length_s=5 instruct the pipeline to execute
# the sliding window internally with optimized memory reuse.
# batch_size=1 enforces low memory usage. Increase slightly (e.g., 2 or 4) only if VRAM allows.
result = pipe(
    file_path,
    chunk_length_s=15,
    stride_length_s=5,
    batch_size=1,
    return_timestamps="word",
    generate_kwargs=generate_kwargs
)

# 7. Format the structural word-level results cleanly
formatted_word_timestamps = []
for chunk in result["chunks"]:
    if chunk["timestamp"] is not None:
        formatted_word_timestamps.append({
            "word": chunk["text"].strip(),
            "start": round(chunk["timestamp"][0], 2),
            "end": round(chunk["timestamp"][1], 2)
        })

print("\n--- FULL TRANSCRIPTION ---")
print(result["text"].strip())

print("\n--- EXTRACTED WORD TIMESTAMPS (FIRST 20) ---")
import pprint
pprint.pprint(formatted_word_timestamps[:20])

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Processing with internal pipeline chunking (Memory Safe)...



In [16]:
outputs['segments']

[[{'start': tensor(0., dtype=torch.float64),
   'end': tensor(30., dtype=torch.float64),
   'tokens': tensor([50364,  4960,   439,  1412,   680,  5623,    11,   342,  8997,  2750,
           16235, 23475,  4960,   472,  6889,   680,  5623,    11,   293,  8382,
              12,    65,   852, 16235, 23475,  4960,   257,  1359, 25993,   680,
            5623,    13,   682,  3124,    11,  8382,    12,    65,   852, 16235,
           23475,   307,  2673, 16494,   570,   309,  6417,   257,   665,  4772,
            1296, 28270, 10493,   293,  3097, 11826,    13]),
   'idxs': (3, 60),
   'result': {'sequences': tensor([50258, 50259, 50360, 50364,  4960,   439,  1412,   680,  5623,    11,
              342,  8997,  2750, 16235, 23475,  4960,   472,  6889,   680,  5623,
               11,   293,  8382,    12,    65,   852, 16235, 23475,  4960,   257,
             1359, 25993,   680,  5623,    13,   682,  3124,    11,  8382,    12,
               65,   852, 16235, 23475,   307,  2673, 16494,   

1. Pacing & Speed FeaturesThese metrics tell you how fast, rushed, or deliberate a speaker's articulation is.
  * Words Per Minute (WPM): Total words divided by the duration of the audio clip.
  * Instantaneous Speaking Rate: Calculated by taking the duration of an individual word ($\text{end} - \text{start}$) or a short phrase window. Rushed words have incredibly tight windows.
  * Articulatory Trajectory (Acceleration/Deceleration): Tracking how WPM fluctuates over the course of the 2 minutes. Does the candidate accelerate when nervous?
2. Pause & Fluency FeaturesTracking the "silence" between the words yields incredible insights into a speaker's hesitation or confidence.
  * Inter-word Silence (Pauses): Calculated by measuring the gap between a word's end timestamp and the next word's start timestamp ($\text{start}_{n+1} - \text{end}_n$).
  * Micro-pauses vs. Gross Pauses:
    - Micro-pauses (under 0.2 seconds) are natural linguistic boundaries.
    - Gross pauses (greater than 0.5 to 2.0+ seconds) indicate cognitive loading, hesitation, or freezing.Speech-to-Pause Ratio: The total amount of time spent actually pronouncing words versus the total time spent in silence.
3. Disfluency & Filler Word Features⚠️ Critical Whisper Warning: Standard OpenAI Whisper is trained to generate clean, readable text. By default, its internal decoder actively strips out filler words like "uh", "um", "ah", or repetitive stammers to optimize transcripts.  If you want to pull filler words out of Whisper, you have to use a workaround:
  * The Prompting Trick: Pass an initial_prompt="Umm, uh, like, okay." to the Whisper processor. This signals the model that it is acceptable to output disfluencies.
  * Alternative Tooling: Use CrisperWhisper or WhisperX (which handles forced phonetic alignment) to ensure filler words aren't dropped.<br>
  Once enabled, you can map:
  * Filler Word Frequency (Disfluency Count): Counting the occurrence of explicit tokens like "um", "uh", "like", "so", or "you know".
  * Filler Injection Rate: The percentage of total words that are filler items.
  * Hesitation Placement: Detecting if fillers appear at structural points (e.g., at the start of a sentence while planning a thought) or mid-sentence (indicating a struggle to find vocabulary).
4. Semantic & Text-Based FeaturesOnce you have the text string itself, you can pass it to downstream NLP tools (like a lightweight BERT model or an LLM) to get semantic features:
  * Vocabulary Diversity (Type-Token Ratio): The ratio of unique words to total words spoken. High diversity usually suggests high fluency and vocabulary command.
  * Sentiment and Tone Shift: Evaluating how the literal meaning of the words changes throughout the response.
  * Readability / Grade Level (Flesch-Kincaid): Assessing structural complexity based on word choice and sentence lengths calculated via the timestamp boundaries.

In [ ]:
# Hypothethical whisper output list
word_timestamps = [
    {"word": "Let's", "start": 0.5, "end": 0.8},
    {"word": "see", "start": 0.9, "end": 1.4},
    {"word": "um", "start": 2.2, "end": 2.6}, # Filler
    {"word": "next", "start": 2.8, "end": 3.1}
]

fillers = ["um", "uh", "ah", "like"]
total_words = len(word_timestamps)

# 1. Calculate Pacing
duration = word_timestamps[-1]["end"] - word_timestamps[0]["start"]
wpm = (total_words / duration) * 60

# 2. Calculate Pauses and Find Fillers
pause_durations = []
filler_count = 0

for i in range(len(word_timestamps) - 1):
    current_word = word_timestamps[i]
    next_word = word_timestamps[i+1]

    # Pause is the gap between current end and next start
    gap = next_word["start"] - current_word["end"]
    if gap > 0:
        pause_durations.append(gap)

    if current_word["word"].lower().strip(".,!?") in fillers:
        filler_count += 1

print(f"Pacing: {wpm:.1f} WPM")
print(f"Total Fillers: {filler_count}")
print(f"Average Pause Duration: {np.mean(pause_durations):.2f}s")